In [30]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import random
import numpy as np

In [31]:
df = pd.read_csv("Faceplate.csv")

print("First 10 transactions:")
display(df.head(10))

support_count = ((df["Red"] == 1) & (df["White"] == 1)).sum()
total_transactions = len(df)
support = support_count / total_transactions

print(f"Support count for {{red, white}}: {support_count}")
print(f"Support for {{red, white}}: {support:.2f} ({support*100:.0f}%)")


First 10 transactions:


,Transaction,Red,White,Blue,Orange,Green,Yellow
0,1,1,1,0,0,1,0
1,2,0,1,0,1,0,0
2,3,0,1,1,0,0,0
3,4,1,1,0,1,0,0
4,5,1,0,1,0,0,0
5,6,0,1,1,0,0,0
6,7,1,0,1,0,0,0
7,8,1,1,1,0,1,0
8,9,1,1,1,0,0,0
9,10,0,0,0,0,0,1


Support count for {red, white}: 4
Support for {red, white}: 0.40 (40%)


In [32]:
df = pd.read_csv("Faceplate.csv")

#droping transaction columns
item_cols = [c for c in df.columns if c.lower() != "transaction"]
basket = df[item_cols].astype(bool)

# 2.1 Frequent itemsets with min support = 0.2
frequent_itemsets = apriori(
    basket,
    min_support=0.2,
    use_colnames=True
).sort_values(["support", "itemsets"], ascending=[False, True]).reset_index(drop=True)

print("Frequent itemsets (min support = 0.2):")
display(frequent_itemsets)

# 2.2 Association rules with min confidence = 0.5, sorted by lift desc
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
).sort_values("lift", ascending=False).reset_index(drop=True)

print("Association rules (min confidence = 0.5), sorted by lift:")
display(rules)

# 2.3 Top 6 rules by lift, selected columns only
top6 = rules.head(6).copy()

# Convert frozensets to readable strings
top6["antecedents"] = top6["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
top6["consequents"] = top6["consequents"].apply(lambda s: ", ".join(sorted(list(s))))

cols_to_show = ["antecedents", "consequents", "support", "confidence", "lift", "leverage"]
top6_output = top6[cols_to_show]

print("Top 6 rules by lift (requested columns):")
display(top6_output)


# 2.4 Translate rule with highest lift into a sentence
best = rules.iloc[0]
ant = ", ".join(sorted(list(best["antecedents"])))
con = ", ".join(sorted(list(best["consequents"])))
conf_pct = best["confidence"] * 100
lift_val = best["lift"]

sentence = (
    f'If {ant} are purchased, then with confidence {conf_pct:.1f}% '
    f'{con} will also be purchased. This rule has a lift ratio of {lift_val:.3f}.'
)

print("Interpretation of highest-lift rule:")
print(sentence)

Frequent itemsets (min support = 0.2):


,support,itemsets
0,0.7,(White)
1,0.6,(Red)
2,0.6,(Blue)
3,0.4,"(White, Red)"
4,0.4,"(Red, Blue)"
5,0.4,"(White, Blue)"
6,0.2,(Orange)
7,0.2,(Green)
8,0.2,"(Red, Green)"
9,0.2,"(Orange, White)"


Association rules (min confidence = 0.5), sorted by lift:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"(White, Red)",(Green),0.4,0.2,0.2,0.500000,2.500000,1.0,0.12,1.600000,1.000000,0.500000,0.375000,0.750000
1,(Green),"(White, Red)",0.2,0.4,0.2,1.000000,2.500000,1.0,0.12,inf,0.750000,0.500000,1.000000,0.750000
2,(Green),(Red),0.2,0.6,0.2,1.000000,1.666667,1.0,0.08,inf,0.500000,0.333333,1.000000,0.666667
3,"(White, Green)",(Red),0.2,0.6,0.2,1.000000,1.666667,1.0,0.08,inf,0.500000,0.333333,1.000000,0.666667
4,(Green),(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
5,(Orange),(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
6,"(Red, Green)",(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
7,(Blue),(Red),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.200000,0.250000,0.500000,0.166667,0.666667
8,(Red),(Blue),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.200000,0.250000,0.500000,0.166667,0.666667
9,(Blue),(White),0.6,0.7,0.4,0.666667,0.952381,1.0,-0.02,0.900000,-0.111111,0.444444,-0.111111,0.619048


Top 6 rules by lift (requested columns):


,antecedents,consequents,support,confidence,lift,leverage
0,"Red, White",Green,0.2,0.5,2.500000,0.12
1,Green,"Red, White",0.2,1.0,2.500000,0.12
2,Green,Red,0.2,1.0,1.666667,0.08
3,"Green, White",Red,0.2,1.0,1.666667,0.08
4,Green,White,0.2,1.0,1.428571,0.06
5,Orange,White,0.2,1.0,1.428571,0.06


Interpretation of highest-lift rule:
If Red, White are purchased, then with confidence 50.0% Green will also be purchased. This rule has a lift ratio of 2.500.


In [33]:
df = pd.read_csv("CharlesBookClub.csv")

# 3.1
exclude_cols = {
    "Seq#", "ID#", "Gender", "M", "R", "F", "FirstPurch", "Related Purchase"
}

# Drop explicitly excluded fields + any code columns
book_cols = [
    c for c in df.columns
    if c not in exclude_cols and "code" not in c.lower()
]

binary_matrix = (df[book_cols] > 0).astype(int)

print("First 10 rows of binary incidence matrix:")
display(binary_matrix.head(10))


# 3.2 Apriori with minimum support = 200 transactions
min_support = 200 / len(binary_matrix)   # absolute threshold of 200 transactions

frequent_itemsets = apriori(
    binary_matrix.astype(bool),
    min_support=min_support,
    use_colnames=True
).sort_values("support", ascending=False).reset_index(drop=True)

print(f"Number of frequent itemsets found: {len(frequent_itemsets)}")
display(frequent_itemsets)


# 3.3 Association rules with minimum confidence = 0.5
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
).sort_values("lift", ascending=False).reset_index(drop=True)

top25 = rules.head(25).copy()

# Make itemsets readable
top25["antecedents"] = top25["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
top25["consequents"] = top25["consequents"].apply(lambda s: ", ".join(sorted(list(s))))

result = top25[["antecedents", "consequents", "support", "confidence", "lift", "leverage"]]

print("Top 25 rules by lift:")
display(result)



First 10 rows of binary incidence matrix:


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence,Yes_Florence,No_Florence
0,0,1,1,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,0,0,1
2,1,1,1,0,1,0,1,1,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,0,0,1
5,0,0,0,0,0,0,0,0,0,0,0,0,1
6,0,0,0,0,0,0,1,0,0,0,0,0,1
7,1,0,0,0,0,0,0,0,0,0,0,0,1
8,0,0,0,0,0,0,0,0,0,0,0,0,1
9,0,0,1,0,0,0,0,0,0,0,0,0,1


Number of frequent itemsets found: 116


,support,itemsets
0,0.91550,(No_Florence)
1,0.41550,(CookBks)
2,0.39400,(ChildBks)
3,0.38075,"(CookBks, No_Florence)"
4,0.35525,"(ChildBks, No_Florence)"
...,...,...
111,0.05250,"(CookBks, DoItYBks, ChildBks, No_Florence, Geo..."
112,0.05150,"(CookBks, YouthBks, ArtBks)"
113,0.05100,"(GeogBks, ChildBks, ArtBks)"
114,0.05100,"(RefBks, ChildBks, GeogBks, No_Florence)"


Top 25 rules by lift:


,antecedents,consequents,support,confidence,lift,leverage
0,Yes_Florence,Florence,0.08450,1.000000,11.834320,0.077360
1,Florence,Yes_Florence,0.08450,1.000000,11.834320,0.077360
2,"RefBks, YouthBks","ChildBks, CookBks",0.05525,0.680000,2.809917,0.035588
3,"DoItYBks, RefBks","ChildBks, CookBks, No_Florence",0.05600,0.605405,2.770734,0.035789
4,"DoItYBks, No_Florence, RefBks","ChildBks, CookBks",0.05600,0.662722,2.738520,0.035551
5,"DoItYBks, RefBks","ChildBks, CookBks",0.06125,0.662162,2.736207,0.038865
6,"DoItYBks, YouthBks","ChildBks, CookBks",0.06700,0.648910,2.681448,0.042014
7,"DoItYBks, No_Florence, YouthBks","ChildBks, CookBks",0.06000,0.648649,2.680366,0.037615
8,"DoItYBks, YouthBks","ChildBks, CookBks, No_Florence",0.06000,0.581114,2.659560,0.037440
9,"GeogBks, RefBks","ChildBks, CookBks",0.05025,0.614679,2.539995,0.030467


### 4.1 rule with the highest support value is (CookBks, No_Florence)

-Antecedents: CookBks
-support: 0.38075
-confidence: 0.91637
-lift ratio: 1.001

### 4.2 highest lift ratio from the book purchase rules is (Yes_Florence - Florence) with support 0.38075

-Highest support rule support: 0.38075
-highest lift rule support: 0.08450

Trade-off:

The highest-lift rule is very strong/efficient (very non-random association), but it affects a much smaller portion of transactions.

The highest-support rule is less “surprising” (lift near 1), but applies to many more transactions, so it is often more useful for broad business impact.

### 4.3 rule with lowest confidence is

{doltYBks, YouthBks}-{ChildBks,CookBks,NoFlorence} with support: 0.06000, confidence: 0.581114, lift: 2.659560



In [34]:
# 5.1 Synthetic dataset: 50 transactions x 9 items, random.seed(0)
random.seed(0)
np.random.seed(0)

n_transactions = 50
n_items = 9
item_cols = [f"Item{i}" for i in range(1, n_items + 1)]

# Binary incidence matrix (0/1)
data = np.random.randint(0, 2, size=(n_transactions, n_items))
df = pd.DataFrame(data, columns=item_cols)

print("First 10 rows of synthetic binary matrix:")
display(df.head(10))


# 5.2 Apriori with MS = 2 transactions (2/50 = 0.04), CI = 0.7
min_support = 2 / n_transactions
frequent_itemsets = apriori(df.astype(bool), min_support=min_support, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print(f"Number of frequent itemsets: {len(frequent_itemsets)}")
print(f"Number of rules (confidence >= 0.7): {len(rules)}")

# 5.3 Top 6 rules by uplift opportunity (use lift as uplift ratio)
# also add 'uplift' = lift - 1
rules = rules.copy()
rules["uplift_ratio"] = rules["lift"]      # requested sorting metric
rules["uplift"] = rules["lift"] - 1

# readable sets
to_text = lambda s: ", ".join(sorted(list(s)))
rules["antecedents"] = rules["antecedents"].apply(to_text)
rules["consequents"] = rules["consequents"].apply(to_text)

top6 = rules.sort_values("uplift_ratio", ascending=False).head(6)

print("\n Top 6 rules by uplift ratio (descending):")
display(top6[["antecedents", "consequents", "support", "confidence", "uplift_ratio", "uplift"]])

# short interpretation
if len(top6) > 0 and top6["uplift_ratio"].max() >= 2:
    print("Yes: at least one rule shows exceptionally high uplift ratio (>= 2), despite random data.")
else:
    print("No exceptionally high uplift ratio found (using threshold >= 2).")

First 10 rows of synthetic binary matrix:


,Item1,Item2,Item3,Item4,Item5,Item6,Item7,Item8,Item9
0,0,1,1,0,1,1,1,1,1
1,1,1,0,0,1,0,0,0,0
2,0,1,0,1,1,0,0,1,1
3,1,1,0,1,0,1,0,1,1
4,0,1,1,0,0,1,0,1,1
5,1,1,1,0,1,0,1,1,1
6,1,0,1,0,0,1,1,0,1
7,0,1,0,0,0,0,0,1,1
8,0,0,0,1,1,0,1,0,0
9,1,0,1,1,1,1,1,1,0


Number of frequent itemsets: 358
Number of rules (confidence >= 0.7): 377

 Top 6 rules by uplift ratio (descending):


,antecedents,consequents,support,confidence,uplift_ratio,uplift
376,"Item6, Item7, Item8, Item9","Item2, Item3, Item5",0.04,1.0,5.555556,4.555556
374,"Item2, Item3, Item5, Item6, Item9","Item7, Item8",0.04,1.0,5.000000,4.000000
359,"Item3, Item4, Item5, Item6, Item7","Item1, Item8",0.04,1.0,5.000000,4.000000
327,"Item2, Item4, Item5, Item7","Item6, Item8",0.04,1.0,4.545455,3.545455
347,"Item1, Item2, Item3, Item6, Item8","Item4, Item7",0.04,1.0,4.545455,3.545455
286,"Item1, Item3, Item6, Item8","Item4, Item7",0.06,1.0,4.545455,3.545455


Yes: at least one rule shows exceptionally high uplift ratio (>= 2), despite random data.
